# 00-baseline — 압축 없이 통과시키기

**아무것도 압축하지 않는 랩입니다.** 그래서 가장 먼저 만들었습니다.

두 가지 역할을 합니다.

| 역할 | 뜻 |
|---|---|
| **기준선** | "40% 절감" 이 무엇 대비인지 정해줍니다 |
| **하네스 점검** | 여기서 틀리면 다른 랩을 돌려봐야 소용없습니다 |

압축을 안 했으니 결과가 뻔합니다. **이 값이 안 나오면 압축기가 아니라 하네스가 고장 난 것입니다.**

```
절감률          0.0%
생존 최악       100%
```

이 노트북은 `compress.py` 가 하는 일을 **단계별로 쪼개서** 보여줍니다.
한 번에 돌리려면 터미널에서 이렇게 하면 됩니다.

```bash
python compress.py configs/noop.yaml
```

## 1. kit 불러오기

랩은 `labs/` 를 경로에 넣고 `from kit import ...` 로 씁니다.

**랩끼리는 서로를 import 하지 않습니다.** kit 은 랩이 아니라 기반이라 단방향입니다.

In [ ]:
import sys
from pathlib import Path

LABS = Path.cwd().resolve().parents[0]      # labs/00-baseline -> labs
sys.path.insert(0, str(LABS))

from kit import VERSION, config as C, dataset, metrics, env
from kit.display import table, pct
from kit.runner import Run

# .env 는 labs/.env → 저장소 루트 .env → scripts/explore/.env 순으로 찾습니다.
# 변수명은 scripts/explore/.env.example 과 같습니다.
dotenv = env.load(verbose=True)

print("kit", VERSION)
print("배포명 기본값:", env.get("AZURE_OPENAI_DEPLOYMENT", "(없음)"))
print("엔드포인트    :", env.mask_endpoint(env.get("AZURE_OPENAI_ENDPOINT")))

## 2. 설정 읽기

조건은 코드가 아니라 yaml 에 둡니다. 그래야 **"조건 1개 = 파일 1개"** 가 되고,
`runs/` 경로에 config 이름이 그대로 남습니다.

```yaml
name: noop
lab: 00-baseline
params: {}
dataset:
  path: ../../data/sample
model: gpt-5.4          # 토큰 계산용 인코딩 기준 — API 를 부르지 않습니다
```

In [ ]:
cfg = C.load("configs/noop.yaml")

table(
    ["항목", "값"],
    [["name", cfg.name],
     ["lab", cfg.lab],
     ["params", cfg.params or "(없음)"],
     ["dataset.path", cfg.dataset.get("path")],
     ["model", cfg.model],
     ["tokenizer", cfg.tokenizer or "(기본: local)"]],
    align=["left", "left"],
    title="설정",
)

## 2-1. 토큰을 어떻게 셀 것인가

**두 가지 방식이 있고, 설정으로 고릅니다.**

| | `local` | `api` |
|---|---|---|
| 방법 | tiktoken (없으면 문자 근사) | 모델을 호출해 `usage.input_tokens` |
| 비용 | 0 | 텍스트마다 호출 1회 |
| 정확도 | 근사 | **과금 기준 그대로** |
| 포함되는 것 | 텍스트만 | 텍스트 + **메시지 포맷 오버헤드** |

```yaml
tokenizer:
  mode: api            # local | api
  deployment: gpt-5.4  # 생략하면 AZURE_OPENAI_DEPLOYMENT
  cache: true          # 같은 텍스트는 한 번만 호출
```

**`api` 는 캐시가 필수입니다.** 케이스 N건이면 압축 전후로 2N 회를 부르고,
압축률 스윕을 10단계 돌리면 그만큼 곱해집니다.
같은 텍스트의 결과는 디스크(`kit/.cache/`)에 남아 다음 실행에서 재사용됩니다.

> **주의** — `api` 값에는 메시지 포맷 오버헤드가 포함됩니다(보통 +6 토큰).
> 압축 전후를 같은 방식으로 재므로 **비율 비교에는 문제가 없지만**,
> `local` 값과 나란히 놓으면 안 됩니다. 그래서 `token_backend` 를 기록합니다.

In [ ]:
# 여기서 방식을 바꿔 볼 수 있습니다. config 의 tokenizer 를 덮어씁니다.
USE_API = False        # True 로 바꾸면 실제 모델을 호출해 실측합니다

spec = dict(cfg.tokenizer)
if USE_API:
    spec = {"mode": "api",
            "deployment": spec.get("deployment") or env.get("AZURE_OPENAI_DEPLOYMENT"),
            "cache": True}

counter = metrics.T.make_counter(spec, cfg.model)
print("측정 방식:", counter.backend)

sample = cases[0].text if "cases" in dir() else "환불 수수료는 결제금액의 10%입니다."
print(f"예시 {len(sample)}자 → {counter(sample):,} 토큰")

## 3. 코퍼스 살펴보기

한 줄이 한 케이스인 jsonl 입니다.

```json
{"id": "doc-001", "text": "...", "question": "...",
 "must_include": ["10%", "면제"], "meta": {"kind": "numeric"}}
```

**`must_include`** 가 핵심입니다. 압축 후에도 이 문자열이 남아 있어야 정답을 말할 수 있습니다.
LLM 을 부르지 않으므로 스윕을 수백 번 돌려도 비용이 0 입니다.

In [ ]:
cases = dataset.load(cfg.dataset["path"], limit=cfg.dataset.get("limit"))
info = dataset.summarize(cases)

table(
    ["항목", "값"],
    [["케이스", f"{info['n_cases']}건"],
     ["전체 문자수", f"{info['n_chars']:,}자"],
     ["must_include 있는 케이스", f"{info['with_must_include']}건"],
     ["유형", ", ".join(f"{k} {v}" for k, v in info["kinds"].items())]],
    align=["left", "left"],
    title=f"코퍼스 · {Path(cfg.dataset['path']).name}",
)

table(
    ["id", "유형", "글자", "must_include", "원문 앞부분"],
    [[c.id, c.kind, f"{len(c.text):,}", ", ".join(c.must_include)[:26],
      c.text[:34].replace(chr(10), " ") + "…"] for c in cases[:6]],
    align=["left", "left", "right", "left", "left"],
    title="앞 6건",
    note="유형은 겨냥하는 실패 모드입니다. numeric=숫자 절단, negation=부정어 소실, "
         "identifier=ID 파편화, similar=엉뚱한 문서 생존, structured=구조 파괴, short=압축이 손해.",
)

## 4. 압축 — 이 랩은 그대로 통과시킵니다

다른 랩도 **같은 시그니처**를 씁니다. 그래야 같은 하네스로 비교할 수 있습니다.

```python
def compress(text: str, **params) -> tuple[str, dict]:
    return compressed_text, extra_meta
```

In [ ]:
def compress(text: str, **params):
    """압축하지 않습니다."""
    return text, {}


c = cases[0]
after, _ = compress(c.text, **cfg.params)

print(f"[원문 {len(c.text)}자]")
print(" ", c.text[:80], "…")
print(f"\n[압축 후 {len(after)}자]")
print(" ", after[:80], "…")
print(f"\n동일한가: {c.text == after}")

## 5. 케이스별 지표

두 축만 봅니다.

| 지표 | 뜻 |
|---|---|
| `saved` | 얼마나 줄었나 |
| `survival` | 정답 문자열이 살아남은 비율 |

압축을 안 했으니 `saved` 는 0, `survival` 은 1 이어야 합니다.

In [ ]:
records = [
    metrics.per_case(c.id, c.kind, c.text, compress(c.text, **cfg.params)[0],
                     c.must_include, counter)
    for c in cases
]

table(
    ["id", "유형", "토큰 전", "토큰 후", "절감", "정답 보존율"],
    [[r["id"], r["kind"], f"{r['tokens_before']:,}", f"{r['tokens_after']:,}",
      pct(r["saved"]), pct(r["survival"])] for r in records],
    align=["left", "left", "right", "right", "right", "right"],
    title="케이스별",
    note="정답 보존율 = must_include 문자열 중 압축 후에도 남아 있는 비율입니다.",
)

## 6. 집계 — 평균만 보면 안 됩니다

**정답 보존율**은 각 케이스에서 `must_include` 문자열 중 압축 후에도 남은 비율입니다.
100% 면 하나도 안 잃은 것입니다.

집계는 평균과 함께 **최저값**과 **유형별 분해**를 항상 냅니다.

> 평균 90% 여도 한 케이스가 0% 면 그 질문에는 아예 답할 수 없습니다.
> 평균은 그걸 가립니다. **최저 보존율부터 보세요.**

In [ ]:
m = metrics.aggregate(records, counter)

table(
    ["지표", "값", "정상값", "뜻"],
    [["절감률", pct(m["saved"]), "0.0%", "압축을 안 했으므로"],
     ["토큰", f"{m['tokens_before']:,} → {m['tokens_after']:,}", "변화 없음", ""],
     ["평균 보존율", pct(m.get("survival_mean")), "100%", "전체 케이스 평균"],
     ["하위 5%", pct(m.get("survival_p5")), "100%", "나쁜 쪽 5% 지점"],
     ["최저 보존율", pct(m.get("survival_worst")), "100%", "가장 많이 깨진 케이스"],
     ["온전한 케이스", pct(m.get("survived_all_rate")), "100%", "하나도 안 잃은 비율"],
     ["측정 방식", m["token_backend"], "local 또는 api", "다르면 비교 불가"]],
    align=["left", "right", "right", "left"],
    title="집계",
)

table(
    ["유형", "건수", "절감", "평균 보존율", "최저 보존율"],
    [[k, v["n"], pct(v["saved"]), pct(v["survival_mean"]), pct(v["survival_worst"])]
     for k, v in m["by_kind"].items()],
    align=["left", "right", "right", "right", "right"],
    title="유형별",
    note="어느 유형이 먼저 무너지는지 보려고 나눕니다. 지금은 압축을 안 했으니 전부 100% 입니다.",
)

## 7. 하네스 자가 점검

압축을 안 했으니 결과가 정해져 있습니다. **틀리면 어디가 고장 났는지 알려줍니다.**

In [ ]:
problems = []
if m["saved"] != 0.0:
    problems.append(f"절감률이 0 이 아닙니다 ({m['saved']:.2%}) — 로더가 원문을 바꾸고 있습니다")
if m.get("survival_worst") not in (None, 1.0):
    problems.append(f"최저 보존율이 100% 가 아닙니다 ({m['survival_worst']:.1%}) — "
                    f"must_include 나 정규화를 확인하세요")

if problems:
    print("하네스 점검 실패")
    for p in problems:
        print("  ✗", p)
else:
    print("하네스 정상 — 다른 랩을 돌려도 됩니다.")

if hasattr(counter, "stats") and counter.stats():
    counter.save()
    print("토큰 카운터:", counter.stats())

## 8. 결과 기록

`runs/<lab>/<config>/<타임스탬프>/` 아래 네 개가 남습니다.

| 파일 | 내용 | 커밋 |
|---|---|---|
| `config.snapshot.yaml` | 설정 + **kit 버전** + 환경 | O |
| `metrics.json` | 집계 지표 | O |
| `report.md` | 사람이 읽는 요약 | O |
| `records.jsonl` | 케이스별 원문·압축문 | **X** |

`records.jsonl` 만 제외하는 이유는 용량도 있지만 **원문이 그대로 들어가기** 때문입니다.
대신 압축 전후를 둘 다 남겨서 "왜 이 케이스가 깨졌나" 를 나중에 다시 볼 수 있습니다.

kit 버전을 박는 이유는 **지표 정의가 바뀌어도 과거 숫자를 해석할 수 있게** 하기 위해서입니다.

In [ ]:
run = Run(cfg, LABS.parent / "runs")
for c in cases:
    after, extra = compress(c.text, **cfg.params)
    run.add(metrics.per_case(c.id, c.kind, c.text, after, c.must_include, counter, extra),
            before=c.text, after=after)

out = run.finish(m, ["압축 없음. 다른 랩의 절감률은 이 결과를 기준으로 읽습니다.",
                     f"토큰 측정 방식: {counter.backend}"])

print("기록 위치:", out)
for f in sorted(out.iterdir()):
    print(f"  {f.name:24s} {f.stat().st_size:>7,}B")
print("\n─── report.md ───")
print((out / "report.md").read_text(encoding="utf-8"))

## 정리

- **기준선이 없으면 절감률은 의미가 없습니다** — 무엇 대비인지가 있어야 합니다
- **하네스를 먼저 검증합니다** — `kit` 을 고친 뒤에는 항상 여기부터 돌리세요
- **정답 보존율은 평균 대신 최저값** — 평균은 한 케이스의 붕괴를 가립니다
- **측정 방식을 기록합니다** — `local` 과 `api` 는 값이 다르므로 섞으면 안 됩니다

### 다음 랩

`01-lossless-structure` 입니다. `compress()` 만 바꾸면 나머지는 그대로 재사용됩니다.

```python
def compress(text, **params):
    return to_table(json.loads(text)), {"method": "json→table"}
```